# 📄 Módulo 5: Resumen Inteligente de Comunicados Oficiales (NLP)
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Sintetizar comunicados oficiales extensos del **ENFEN** y **SENAMHI** a fichas ejecutivas claras para productores agropecuarios mediante el algoritmo no supervisado **TextRank con matrices TF-IDF**.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Librerías importadas correctamente.")


### 1. Carga del Comunicado Oficial ENFEN N° 04-2026


In [ ]:
doc_path = '../data/comunicados_enfen/comunicado_enfen_n04_2026.txt'
with open(doc_path, 'r', encoding='utf-8') as f:
    texto = f.read()

print(f"Total de caracteres en el documento original: {len(texto)}")
print("\n--- Vista previa del texto ---")
print(texto[:400] + "...")


### 2. Preprocesamiento: Limpieza y Segmentación de Oraciones


In [ ]:
clean_text = re.sub(r'\s+', ' ', texto).strip()
oraciones = [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9ÁÉÍÓÚÑ¿¡])', clean_text) if len(s.strip()) > 35]

print(f"Total de oraciones identificadas: {len(oraciones)}")
for i, o in enumerate(oraciones[:3]):
    print(f"[{i+1}] {o}")


### 3. Matriz TF-IDF y Grafo de Similitud Coseno (TextRank)


In [ ]:
stopwords_es = ['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo']
vectorizer = TfidfVectorizer(stop_words=stopwords_es)
tfidf_mat = vectorizer.fit_transform(oraciones)

# Matriz de adyacencia del grafo (similitud coseno)
sim_matrix = cosine_similarity(tfidf_mat, tfidf_mat)
np.fill_diagonal(sim_matrix, 0)

# Iteración del algoritmo PageRank
n = sim_matrix.shape[0]
row_sums = sim_matrix.sum(axis=1)
norm_matrix = np.divide(sim_matrix, row_sums[:, np.newaxis], where=row_sums[:, np.newaxis] != 0)

scores = np.ones(n) / n
for _ in range(30):
    scores = (1 - 0.85) / n + 0.85 * norm_matrix.T.dot(scores)

top_indices = sorted(np.argsort(scores)[::-1][:4])
resumen = [oraciones[i] for i in top_indices]

print("📋 Ficha Ejecutiva Generada para Agricultores:")
for s in resumen:
    print("•", s)


### 4. Evaluación de Métricas de Compresión Textual


In [ ]:
len_original = len(texto)
len_resumen = len(" ".join(resumen))
ratio_compresion = (1.0 - (len_resumen / len_original)) * 100

print("📊 Métricas de Compresión:")
print(f"• Caracteres Originales: {len_original}")
print(f"• Caracteres del Resumen: {len_resumen}")
print(f"• Porcentaje de Compresión: {ratio_compresion:.1f}%")
print(f"• Reducción de Oraciones: {len(oraciones)} ➡️ {len(resumen)}")
